# An overview of Machine Learning
(at least the non-deep-learning part)

In his online lectures for the course "Statistical Rethinking", professor Richard McElreath references the story of "Golem", an autamaton in Jewish folklore. Golem was created from clay but given the power to carry out tasks based on specific commands. 

This 2-3 thousand year old story shows humanity's desire to create a brain; or at least a mechanism which can act independently of the human mind.

In this class we will limit our scope to a very specific set of algorithms that learn (aka "machine learning"): algorithms which operate on tabular data.

### A table of data

Let's start with a table of data

#### Supervised learning

<img src="images/table1.png" width="300" />

Finding the relationship between one column and the rest can be considered supervised learning

<img src="images/table_supervised.png" width="300" />

If the target variable is continuous, it is called a **regression** problem and if the target variable is categorical, it is called **classification**.

Some examples of regression algorithms are `Ordinary Least Squares`, `Ridge/Lasso regression`, etc. Some examples of classification algorithms are `Support Vector Machines`, `Logistic Regression`, etc. Algorithms with names like `Random Forest`, `Decision Trees`, etc. have variations that can be used for regression or classification.

#### Unsupervised learning

Sometimes we are not looking to find a relationship among columns, we want to reduce the number of columns (perhaps by finding columns that are too closely related). Such an operation is called **"dimentionality reduction"** and some related algorithms are `PCA`, `UMAP`, `T-SNE`, etc.

<img src="images/table_dim_reduction.png" width="300" />

A more common example of unsupervised learning involves **clustering** similar rows together. An example of such an algorithm is "client segmentation" in marketing where clients are divided into common cohorts. Some common clustering algorithms are `KMeans`, `DB Scan`

<img src="images/table_clustering.png" width="300" />

## Let's prepare data for modeling: various attributes per symbol

In [ ]:
#!pip install humanize

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import humanize
import hvplot.pandas  # noqa


#### Load OHLC price data for all US stocks and another dataset containing sectors for those stocks

In [ ]:
DATA_DIR = Path("../../datasets/market_data")

ohlcv = pd.read_csv(DATA_DIR / "ohlcv_2025-sept.csv.zip", parse_dates=["date"])
sectors = (pd.read_csv(DATA_DIR / "sectors.csv.zip")
             .rename(columns={"company name": "company", "market cap": "market_cap"})
             [["ticker", "company", "sector", "market_cap"]]
             .dropna(subset=["sector"]))


In [ ]:
ohlcv.head()

In [ ]:
sectors.head()

#### Calculate daily returns, dollar volumen and intraday range, per stock

In [ ]:
df = ohlcv.sort_values(["ticker", "date"]).copy()
df["ret"]        = df.groupby("ticker")["close"].pct_change()        # daily return
df["dollar_vol"] = df["close"] * df["volume"]                        # $ traded
df["range_pct"]  = (df["high"] - df["low"]) / df["close"]            # intraday range

df.head()

#### Prepare to calculate "beta" by calculating market wide returns vs stock specific returns

In [ ]:
# equal-weight market return each day → lets us measure each stock's beta
average_market_return_df = df.groupby("date")["ret"].mean().rename("mkt")
average_market_return_df.head()

In [ ]:
df = df.join(average_market_return_df, on="date")
df.head()

Notice that the `mkt` column has unique values per date, not per ticker

In [ ]:
df.loc[df.date == "2025-09-03"].head()

#### Split data into tickers to calculate symbol specific metrics

In [ ]:
grouped_by_ticker_df = df.groupby("ticker")
grouped_by_ticker_df
print(len(grouped_by_ticker_df), df.ticker.nunique())

In [ ]:
grouped_by_ticker_df["ret"].mean()

In [ ]:
features_df = pd.DataFrame({
    "n_days":         grouped_by_ticker_df.size(),
    "mean_ret":       grouped_by_ticker_df["ret"].mean(),                               # typical daily return
    "vol":            grouped_by_ticker_df["ret"].std(),                                # daily volatility
    "total_ret":      grouped_by_ticker_df["close"].last() / grouped_by_ticker_df["close"].first() - 1,    # month return (momentum)
    "avg_range":      grouped_by_ticker_df["range_pct"].mean(),                         # how wild intraday
    "max_abs_move":   grouped_by_ticker_df["ret"].apply(lambda s: s.abs().max()),       # biggest single-day jump
    "avg_dollar_vol": grouped_by_ticker_df["dollar_vol"].mean(),                        # liquidity
    "last_price":     grouped_by_ticker_df["close"].last(),
    "beta":           df.groupby("ticker")[["ret", "mkt"]]
                        .apply(lambda x: x["ret"].cov(x["mkt"]) / x["mkt"].var()),
})

features_df.head()

#### Keep a clean, liquid universe so every demo is meaningful (and fast)

In [ ]:
features_df = features_df[(features_df.n_days >= 15) & (features_df.last_price >= 5)]
features_df = features_df.sort_values("avg_dollar_vol", ascending=False).head(600)
features_df = features_df.join(sectors.set_index("ticker")[["company", "sector", "market_cap"]], how="inner")
features_df = features_df[features_df.market_cap > 0].dropna(subset=["sector", "vol", "beta"])

#### A dataset containing useful metrics per symbol!

In [ ]:
features_df

## Let's do some modeling

In [ ]:
from sklearn.model_selection import train_test_split

### Supervised learning: classify if a company is small, mid or large cap

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#### Try to predict the tier of a stock, given their volume, average return, etc.

In [ ]:
# "Discretize variable into equal-sized buckets based on rank or based on sample quantiles"
features_df["cap_tier"] = pd.qcut(features_df["market_cap"], 3, labels=["small", "mid", "large"])
features_df["cap_tier"]

#### Break up the data into test/train 
You should hide some data from the model, otherwise it might "memorize" everything

In [ ]:
FEATURES = ["mean_ret", "vol", "total_ret", "avg_range",
            "max_abs_move", "avg_dollar_vol", "last_price", "beta"]

X_train, X_test, y_train, y_test = train_test_split(features_df[FEATURES], features_df["cap_tier"], test_size=0.30, random_state=0, stratify=features_df["cap_tier"])

print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

#### Build a random forest classifier to predict the market cap tier of a stock

In [ ]:
%%time
clf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_train, y_train)

#### Calculate accuracy (on data the model has NOT seen before!)

In [ ]:
acc = accuracy_score(y_test, clf.predict(X_test))

print(classification_report(y_test, clf.predict(X_test)))

### Supervised learning: predict the market cap of a stock from inputs (aka regression)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

#### Potential problem: market cap is very skewed

In [ ]:
features_df['market_cap'].sort_values().apply(humanize.intword)

In [ ]:
features_df['market_cap'].plot.hist()

#### so let's take the log of it

In [ ]:
prices_np = np.log10(features_df['market_cap'])
prices_np.plot.hist()

#### Split data into test/train (to test the model on unseen data)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features_df[FEATURES], prices_np, test_size=0.30, random_state=0)

#### Build a regression model

In [ ]:
%%time
reg = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)

#### For root mean squared error (RMSE), lower is better and R², higher is better


In [ ]:
pred = reg.predict(X_test)

print(f"RMSE (log10 cap): {root_mean_squared_error(y_test, pred):.3f}")
print(f"R²              : {r2_score(y_test, pred):.3f}")

#### Btw, what were the most important features for the model?

In [ ]:
pd.Series(reg.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3).to_frame("importance")

### Supervised learning: Can we forecast future values based on the past (time-series forecasting)?
We will load a different dataset for this section, one that is more appropriate for time-series forecasting. 

#### Let's load tick by tick trades for AAPL and MSFT (although we will only keep AAPL).

In [ ]:
ticks = pd.read_csv(DATA_DIR / "trades_2025-09-10_AAPLMSFT_sorted.csv.gz",
                    usecols=["ticker", "price", "size", "sip_timestamp"])

ticks = ticks[(ticks.price > 0) & (ticks.size > 0)]
ticks["ts"] = pd.to_datetime(ticks["sip_timestamp"], unit="ns", utc=True)

#### Convert tick data to 1 minute bars
(and filter out data outside market hours)

In [ ]:
bars = (ticks[ticks.ticker == "AAPL"].set_index("ts")
              .between_time("13:30", "20:00")["price"]      # US regular hours, in UTC
              .resample("1min").last().dropna())            # Convert tick data to 1-minute bars

bars.head()

#### Let's build a forecasting model (but not in scikit-learn)
We will use the AEON library, which is a derivative of the sk-time library, which is modeled on the scikit-learn library

In [ ]:
from aeon.forecasting import NaiveForecaster, RegressionForecaster
from sklearn.linear_model import LinearRegression

#### We can't use a random test/train split, the split has to be time based

In [ ]:
prices_np = bars.to_numpy(float)
horizon = 30                                       # hold out the final 30 minutes — the "future"
train = prices_np[:-horizon]

#### Build a naive forecaster and a regression forecaster

In [ ]:
%%time
# baseline: "the next minute looks like the last one" (a random-walk forecast)
naive_model = NaiveForecaster(strategy="last").iterative_forecast(train, prediction_horizon=horizon)

# model: predict the next minute from the last 10 minutes — regression on a sliding window
reg_model = RegressionForecaster(window=10, regressor=LinearRegression()
                           ).iterative_forecast(train, prediction_horizon=horizon)

In [ ]:
naive_model

In [ ]:
reg_model

#### Calculate performance

In [ ]:
prices_np[-horizon:]

In [ ]:
print(f"RMSE  naive (last value)   : {root_mean_squared_error(prices_np[-horizon:], naive_model):.3f}")
print(f"RMSE  RegressionForecaster : {root_mean_squared_error(prices_np[-horizon:], reg_model):.3f}")

#### Show a plot of the predicted values (and values leading up to the prediction)

In [ ]:
import matplotlib.pyplot as plt

# one labeled frame; each column is NaN where that series doesn't apply
forecasts = pd.DataFrame(index=bars.index)
forecasts["train (the past)"]         = pd.Series(train,  index=bars.index[:-horizon])
forecasts["actual future (held out)"] = pd.Series(prices_np[-horizon:], index=bars.index[-horizon:])
forecasts["naive (last value)"]       = pd.Series(naive_model,  index=bars.index[-horizon:])
forecasts["RegressionForecaster"]     = pd.Series(reg_model,    index=bars.index[-horizon:])

forecasts

In [ ]:
ax = forecasts.plot(figsize=(11, 4.5), style=["-", "-", "--", "--"],
                    color=["0.6", "black", "tab:orange", "tab:green"],
                    title="Forecasting AAPL 1-min price — train on the past, predict the future")
ax.axvline(bars.index[-horizon], color="crimson", ls=":")     # train / test split
ax.set_xlabel("time (UTC)"); ax.set_ylabel("price")
plt.tight_layout(); plt.show()

### Unsupervised learning: anomaly detection - which rows are "different" from the rest?
Btw, anomaly detection is not commonly described in books. IsolationForest is not a commonly used library in scikit-learn.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [ ]:
Z = StandardScaler().fit_transform(features_df[FEATURES])          # standardize: features are on wild scales
iso_model = IsolationForest(contamination=0.02, random_state=0)
features_df["anomaly_score"] = iso_model.fit(Z).decision_function(Z)     # lower = stranger
features_df["is_anomaly"]    = iso_model.predict(Z) == -1

features_df.head()

In [ ]:
features_df["anomaly_score"].plot.hist()

#### Let's take a look at the outliers

Number of outliers among 493 stocks

In [ ]:
features_df.is_anomaly.sum()

In [ ]:
# The columns to display
cols_to_show = ["company", "sector", "vol", "total_ret", "max_abs_move", "last_price"]

# The columns to color (must be numeric)
numeric_cols = ["vol", "total_ret", "max_abs_move", "last_price"]

outliers_df = (features_df[features_df.is_anomaly]
               .sort_values("anomaly_score")
               [cols_to_show]
               .round(3))

outliers_df.style.background_gradient(cmap='Reds', subset=numeric_cols)

#### Let's visually see the outliers in a scatter plot

In [ ]:
features_df.hvplot.scatter(x='vol', y='total_ret', color='is_anomaly', cmap={True: 'red', False: 'green'})

### Unsupervised learning: clustering - which rows are "similar" to each other?

In [ ]:
from sklearn.cluster import KMeans

#### Remove anomalies and cluster the remaining rows

In [ ]:
clean = features_df[~features_df.is_anomaly].copy()

Scale the features so all columns are the same magnitude

In [ ]:
#clean_scaled = StandardScaler().fit_transform(clean[FEATURES])

In [ ]:
clean_scaled_df = pd.DataFrame(StandardScaler().fit_transform(clean[FEATURES]),
                  index=clean.index, columns=FEATURES)
clean_scaled_df

#### Build the clustering model

In [ ]:
clean["cluster"] = KMeans(n_clusters=5, random_state=0, n_init=10).fit_predict(clean_scaled_df)

clean.head()

#### Find median values for each cluster

In [ ]:
profile = clean.groupby("cluster").agg(
    n=("vol", "size"), volatility=("vol", "median"), beta=("beta", "median"),
    dollar_vol=("avg_dollar_vol", "median"), price=("last_price", "median"),
    momentum=("total_ret", "median"))

In [ ]:
profile.style.format("{:,.3f}").background_gradient(cmap='Reds', subset=['n', 'volatility', 'beta', 'dollar_vol', 'price', 'momentum'])

### Unsupervised learning: dimensionality reduction - project a high-dimensional dataset into a lower-dimensional space to visualize it


In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns

In [ ]:
clean_scaled_df

#### Let's reduce the dimensions of the dataset to 2 so we can visualize it

In [ ]:
tsne_embeddings = TSNE(n_components=2, random_state=0, perplexity=30).fit_transform(clean_scaled_df)
viz = pd.DataFrame(tsne_embeddings, columns=["x", "y"], index=clean.index)
viz

#### Add cluster labels to the visualization

In [ ]:
viz["cluster"] = clean["cluster"].astype(str).values
viz

In [ ]:
viz.hvplot.scatter(x="x", y="y", color="cluster")

### Unsupervised learning: dimensionality reduction - can we reduce the number of columns while preserving the information?

In [ ]:
from sklearn.decomposition import PCA

#### Each STOCK is a row; its 20 daily returns are the feature columns

In [ ]:
fingerprints = (df.pivot(index="date", columns="ticker", values="ret")
                  .loc[:, clean.index].dropna(how="all").dropna(axis=1).T)   # stocks × days

returns_scaled = StandardScaler().fit_transform(fingerprints)
fingerprints

#### Reduce dimensions

In [ ]:
pca = PCA().fit(returns_scaled)
cumulative_variance = pd.Series(pca.explained_variance_ratio_.cumsum())
cumulative_variance

In [ ]:
cumulative_variance.plot.bar()

## Acknolwedgement

Some examples and formulas for volatility and returns are sourced from Claude Sonnet and "Python for Finance Cookbook, 2nd Edition"